# Estaciones críticas

En la presentación, los profesores notaron que el modelo **sobreestimaba** y sugirieron ver si un modelo concentrado en las estaciones más críticas ayudaría a planear mejor. Antes de entrenar nada nuevo, este notebook responde dos preguntas, en este orden:

1. **Diagnóstico.** ¿El sesgo del modelo actual se concentra en las estaciones críticas, o está repartido por igual?
2. **Modelo específico.** Si se concentra, ¿un XGBoost entrenado solo con esas estaciones les gana al modelo global **en esas estaciones**? ¿Y mejora el sistema completo si se combinan?

**Qué es "crítica".** Se prueban dos definiciones, las dos calculadas **solo con el periodo de entrenamiento** (si se usara el periodo de prueba, la lista conocería el futuro):

- **Por volumen:** el 10 % de estaciones con más salidas (genera cerca de la cuarta parte de las salidas).
- **Por déficit:** el 10 % de estaciones que más bicicletas pierden por día laboral (llegadas menos salidas) en el pico de la mañana (7 a 9 h) o de la tarde (17 a 19 h). Es la que más se parece a las estaciones que necesitan reposición.

Usa la misma partición, los mismos hiperparámetros y el mismo modelo global que `02_modelo.ipynb`: el RMSE global debe dar cerca de 2.37.

**Cómo correrlo:** Restart & Run All. Necesita `trips_clean.parquet` y `data/processed/busqueda_hiperparametros.csv`. No usa datos nuevos. Tarda unos minutos y necesita unos 3 GB de RAM libres.

**Una advertencia de interpretación.** El modelo predice salidas observadas, no demanda real: si una estación está vacía, no se registran salidas aunque haya personas que quieran una bicicleta. Las estaciones críticas son justo las que más se vacían, así que ahí el error puede tener también esa causa, no solo el modelo.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../src")
import modelo as M
import criticas as C

FRACCION = 0.10                          # qué parte de las estaciones se considera crítica
FRACCIONES_SENSIBILIDAD = [0.05, 0.10, 0.20, 0.30]

## 1. Modelo global (igual que en `02_modelo.ipynb`)

In [ ]:
validos = M.cargar_validos()
estaciones = M.cargar_estaciones()
tabla = M.construir_tabla(validos, estaciones)
del validos

train, test, corte = M.split_por_fechas(tabla, frac_train=0.8)
del tabla
test = test.reset_index(drop=True)
print(f"Train: {train['fecha'].min().date()} a {train['fecha'].max().date()} ({len(train):,} filas)")
print(f"Test:  {test['fecha'].min().date()} a {test['fecha'].max().date()} ({len(test):,} filas)")

mejor = pd.read_csv("../data/processed/busqueda_hiperparametros.csv").iloc[0]
params = {k: (mejor[k] if k == "objective" else int(mejor[k]) if k in ("max_depth", "min_child_weight") else float(mejor[k]))
          for k in M.ESPACIO_BUSQUEDA}
n_arboles = int(mejor["mejor_iteracion"])
print("Hiperparámetros:", params, "| árboles:", n_arboles)

In [ ]:
modelo = M.entrenar_xgb(train, params, n_arboles)
test["pred_xgb"] = np.clip(modelo.predict(test[M.FEATURES]), 0, None)
test["pred_hist"] = M.baseline_promedio(train, test)
test["pred_media4"] = M.baseline_media_4sem(test)
test["_est"] = test["estacion"].astype(str)
del modelo

MODELOS = {"Promedio histórico": "pred_hist", "Media 4 semanas": "pred_media4", "XGBoost (global)": "pred_xgb"}
for nombre, col in MODELOS.items():
    print(f"{nombre:<20s}", {k: round(v, 3) for k, v in M.metricas(test[M.TARGET], test[col]).items()})
print("(XGBoost global debe dar RMSE cercano a 2.37, como en 02_modelo.ipynb; puede diferir en el tercer decimal según la versión de xgboost)")

## 2. ¿Qué estaciones son críticas?

Las dos listas se calculan con el periodo de entrenamiento. El déficit usa los viajes con su estación de **llegada**, así que se lee `trips_clean.parquet` solo con las columnas y fechas necesarias.

In [ ]:
vol = C.volumen_train(train)
crit_vol = C.criticas_por_volumen(vol, FRACCION)

viajes = pd.read_parquet(
    f"{M.PROCESSED_DIR}/trips_clean.parquet",
    columns=["estacion_origen", "estacion_destino", "origen_dt", "destino_dt", "duracion_min"],
    filters=[("origen_dt", ">=", pd.Timestamp(M.FECHA_INICIO).to_pydatetime()), ("origen_dt", "<", corte.to_pydatetime())],
)
viajes = viajes[viajes["duracion_min"].between(1, 45)]
neto = C.flujo_neto_laboral(viajes, vol.index)
del viajes
crit_def, deficit = C.criticas_por_deficit(neto, FRACCION)

nombres = pd.read_csv("../data/processed/stations_geo.csv", dtype={"ce": "string"}).drop_duplicates("ce").set_index("ce")["nombre"]
listas = pd.DataFrame({
    "volumen_train": vol, "deficit_diario_train": deficit.reindex(vol.index),
    "critica_volumen": vol.index.isin(crit_vol), "critica_deficit": vol.index.isin(crit_def),
}).sort_values("volumen_train", ascending=False)
listas["nombre"] = listas.index.map(nombres)
print(f"Estaciones: {len(vol)} | críticas por volumen: {len(crit_vol)} | críticas por déficit: {len(crit_def)} | "
      f"en las dos: {len(crit_vol & crit_def)}")
print(f"Las críticas por volumen generan {100 * vol[list(crit_vol)].sum() / vol.sum():.1f} % de las salidas de entrenamiento; "
      f"las críticas por déficit, {100 * vol[list(crit_def)].sum() / vol.sum():.1f} %.")
display(listas[listas["critica_deficit"]].sort_values("deficit_diario_train", ascending=False)
        [["nombre", "volumen_train", "deficit_diario_train"]].head(10).round(1))

## 3. Diagnóstico: ¿dónde está el sesgo?

Para cada definición se compara el modelo en las críticas y en el resto. `Sesgo` positivo = predice de más. `Sesgo_%_de_demanda` lo expresa contra la demanda media del grupo, para poder comparar grupos con niveles distintos. La última columna dice qué parte del error cuadrático total del modelo cae en cada grupo.

In [ ]:
DEFINICIONES = {"por volumen": crit_vol, "por déficit": crit_def}
diag = C.tabla_grupos(test, DEFINICIONES, MODELOS)
cols = ["definición", "grupo", "modelo", "estaciones", "filas", "demanda_media", "RMSE", "MAE", "Sesgo",
        "Sesgo_%_de_demanda", "% del error cuadrático total"]
display(diag[cols].round(3).style.hide(axis="index"))

### Sesgo por nivel de demanda

Se agrupan las estaciones en 10 deciles según su volumen de entrenamiento (decil 10 = las más usadas). Si el sesgo está concentrado en las estaciones grandes, debería crecer hacia la derecha.

In [ ]:
dec = C.sesgo_por_decil(test, vol, MODELOS)
t_abs = dec.pivot(index="decil", columns="modelo", values="Sesgo")[list(MODELOS)]
t_rel = dec.pivot(index="decil", columns="modelo", values="Sesgo_%_de_demanda")[list(MODELOS)]
dem = dec[dec["modelo"] == "XGBoost (global)"].set_index("decil")["demanda_media"]
display(pd.concat({"Sesgo (salidas por estación-hora)": t_abs, "Sesgo (% de la demanda del decil)": t_rel}, axis=1).round(3))
print("Demanda media por decil:", dem.round(2).tolist())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for nombre in ["Promedio histórico", "XGBoost (global)"]:
    ax[0].plot(t_abs.index, t_abs[nombre], marker="o", label=nombre)
    ax[1].plot(t_rel.index, t_rel[nombre], marker="o", label=nombre)
for a, titulo, yl in ((ax[0], "Sesgo absoluto", "salidas por estación-hora"), (ax[1], "Sesgo relativo", "% de la demanda del decil")):
    a.axhline(0, color="gray", lw=0.8)
    a.set_xticks(range(1, 11))
    a.set_xlabel("Decil de volumen de la estación (10 = más usadas)")
    a.set_ylabel(yl)
    a.set_title(titulo)
    a.legend(fontsize=8)
fig.tight_layout()
fig.savefig("../figures/21_sesgo_por_decil.png", dpi=150)
plt.show()

### Sesgo por hora del día

Los profesores vieron sobreestimación. ¿Ocurre a todas horas o solo en algunas, y es distinto en las críticas?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, (nombre_def, criticas) in zip(ax, DEFINICIONES.items()):
    t = C.sesgo_por_hora(test, criticas, "pred_xgb")
    a.plot(t.index, t["críticas"], marker="o", label="críticas")
    a.plot(t.index, t["resto"], marker="o", label="resto")
    a.axhline(0, color="gray", lw=0.8)
    a.set_title(f"Críticas {nombre_def}")
    a.set_xlabel("Hora del día")
    a.legend(fontsize=8)
ax[0].set_ylabel("Sesgo de XGBoost (salidas por estación-hora)")
fig.tight_layout()
fig.savefig("../figures/22_criticas_sesgo_por_hora.png", dpi=150)
plt.show()

# El mismo sesgo como % de la demanda de esa hora (de 6 a 23 h; de madrugada casi no hay demanda)
rel = pd.concat({nombre_def: C.sesgo_por_hora(test, criticas, "pred_xgb", relativo=True).loc[6:23]
                 for nombre_def, criticas in DEFINICIONES.items()}, axis=1)
display(rel.round(1))

### ¿Cambia con el tamaño del grupo?

Mismo diagnóstico para XGBoost, con la definición por volumen y distintas fracciones de estaciones críticas.

In [ ]:
filas = []
for f in FRACCIONES_SENSIBILIDAD:
    d = C.tabla_grupos(test, {f"top {int(f * 100)} %": C.criticas_por_volumen(vol, f)}, {"XGBoost (global)": "pred_xgb"})
    filas.append(d)
sens = pd.concat(filas)
display(sens[["definición", "grupo", "estaciones", "demanda_media", "RMSE", "Sesgo", "Sesgo_%_de_demanda",
              "% del error cuadrático total"]].round(3).style.hide(axis="index"))

## 4. Modelo específico para las críticas

Se entrena un XGBoost solo con las filas de las estaciones críticas (mismos hiperparámetros; el número de árboles se elige por early stopping dentro de ese subconjunto) y se compara **en esas mismas estaciones y los mismos días de prueba** contra el modelo global y contra el promedio histórico. Después se arma un sistema combinado: modelo específico en las críticas y global en el resto, y se mira el error de todo el sistema.

Las estaciones críticas son pocas (unas 68), así que el modelo específico tiene menos datos que el global: puede ganar por especializarse o perder por falta de datos. Por eso se mide.

In [ ]:
filas = []
test["pred_combinado"] = test["pred_xgb"]
for nombre_def, criticas in DEFINICIONES.items():
    pred, idx, n_arb, n_filas = C.entrenar_especifico(train, test, criticas, params)
    y = test.loc[idx, M.TARGET].to_numpy()
    comparacion = {
        "Promedio histórico": test.loc[idx, "pred_hist"].to_numpy(),
        "XGBoost (global)": test.loc[idx, "pred_xgb"].to_numpy(),
        "XGBoost (solo críticas)": pred,
    }
    for modelo_nombre, p in comparacion.items():
        filas.append({"definición": nombre_def, "modelo": modelo_nombre, **C.resumen(y, p)})
    print(f"{nombre_def}: modelo específico con {n_filas:,} filas de entrenamiento y {n_arb} árboles")

    # Sistema combinado: específico en las críticas, global en el resto
    comb = test["pred_xgb"].to_numpy().copy()
    comb[idx.to_numpy()] = pred
    test[f"pred_comb_{nombre_def}"] = comb
    filas.append({"definición": nombre_def, "modelo": "Sistema completo: global", **C.resumen(test[M.TARGET], test["pred_xgb"])})
    filas.append({"definición": nombre_def, "modelo": "Sistema completo: específico en críticas + global", **C.resumen(test[M.TARGET], comb)})

esp = pd.DataFrame(filas)
display(esp[["definición", "modelo", "filas", "demanda_media", "RMSE", "MAE", "Sesgo", "Sesgo_%_de_demanda"]].round(3).style.hide(axis="index"))

## 5. Guardar resultados

In [ ]:
diag.round(4).to_csv("../data/processed/resultados_criticas_diagnostico.csv", index=False)
esp.round(4).to_csv("../data/processed/resultados_criticas_especifico.csv", index=False)
listas.round(3).to_csv("../data/processed/criticas_listas.csv", index_label="estacion")
print("Guardado: resultados_criticas_diagnostico.csv, resultados_criticas_especifico.csv, criticas_listas.csv")

## Lectura de resultados

Corrida del 10 de noviembre de 2026 (XGBoost 3.2; con otra versión los números pueden cambiar en el tercer decimal). El modelo global dio RMSE 2.368 (2.370 en `02`); el promedio histórico y la media de 4 semanas coinciden con `02`.

**1. El sesgo no se concentra en las estaciones críticas; es proporcional a su demanda.**
- Las críticas por volumen tienen más sesgo en salidas (+0.166 contra +0.047 del resto), pero casi lo mismo como porcentaje de su demanda (2.4 % contra 1.9 %).
- Por decil de volumen el sesgo relativo oscila entre -0.1 % y 3.4 % sin una tendencia clara hacia las estaciones grandes.
- Con la definición por déficit, las críticas se sobreestiman **menos** que el resto (1.3 % contra 2.2 %).

**2. Sí concentran el error, por escala.** Las 68 críticas por volumen (10 % de las estaciones) tienen 36 % del error cuadrático total, y las críticas por déficit, 29 %. Con conteos, el error crece con la demanda, así que no es sorprendente. Con las 136 estaciones del 20 % superior es 51 %.

**3. La sobreestimación es de la noche, no de un tipo de estación.** El sesgo global es positivo de las 19 a las 23 h (+0.21 a las 19 h, +0.42 a las 20 h, +0.37 a las 21 h) y ligeramente negativo en el pico de la mañana (-0.21 a las 8 h). En las críticas por volumen es más grande en salidas (+1.32 a las 20 h) y, como proporción de la demanda de esa hora, algo mayor (19 % contra 13 % a las 20 h), pero de 21 a 23 h es parecida en los dos grupos.

**4. Un modelo solo para las críticas no ayuda.** Con el mismo conjunto de prueba y los mismos hiperparámetros:

| | RMSE en las críticas | Sesgo |
|---|---|---|
| Promedio histórico (volumen) | 5.071 | +0.374 |
| XGBoost global (volumen) | 4.469 | +0.166 |
| XGBoost solo críticas (volumen) | 4.477 | +0.213 |
| XGBoost global (déficit) | 4.002 | +0.063 |
| XGBoost solo críticas (déficit) | 4.006 | +0.093 |

El modelo específico empata con el global en error y tiene un poco más de sesgo; el sistema completo queda en 2.369 contra 2.368. No hay ganancia.

**Conclusión provisional.** La idea de separar las estaciones críticas no mejora la predicción con las variables actuales. Lo que sí se ve es un patrón por hora: el modelo sobreestima de noche (19 a 22 h) y subestima ligeramente en el pico de la mañana. Si se quiere atacar la sobreestimación, el lugar para empezar es la hora y no la estación.

**Qué sigue (hipótesis sin comprobar):**
- Revisar si la sobreestimación nocturna aparece en todos los meses o solo en la prueba (agosto, que es periodo vacacional). La validación mes por mes (`06`) ya tiene el sesgo total de cada mes, pero no por hora.
- Probar una variable de hora por tipo de día o el nivel de demanda reciente a esa hora (por ejemplo, el promedio de las últimas semanas a las 20 h), que el modelo actual solo ve de forma indirecta.

**Limitaciones:**
- Una sola partición de prueba (28 de julio a 31 de agosto); el patrón por hora podría ser del periodo. Falta repetirlo con otros meses.
- Las críticas se definen con el entrenamiento y por fracción de estaciones; con otro umbral las cifras cambian (ver la tabla de sensibilidad), aunque la conclusión sobre el modelo específico no debería depender mucho.
- La demanda observada está censurada en las estaciones que se vacían, y las críticas son las que más se vacían.